# Tuần 4: số liệu baseline phát hiện tấn công OOD

Quy trình: viết mã trên laptop, `git push`, notebook này chỉ `pull` rồi chạy. Kết quả ghi lên Google Drive, không vào repo.

Trước khi chạy: *Runtime → Change runtime type → T4 GPU*.

| Bước | Nội dung |
|---|---|
| 1 | Cài đặt, tải dữ liệu LYCOS-IDS2017 |
| 2 | Kiểm tra dữ liệu và chạy thử nhanh |
| 3 | Tái hiện số P0 của [16] bằng trọng số có sẵn (chỉ suy luận) |
| 4 | P0: 7 mô hình theo giao thức chia ngẫu nhiên |
| 5 | P1: loại từng họ tấn công khỏi tập huấn luyện |
| 6 | Tổng hợp bảng kết quả |

## 1. Cài đặt

In [1]:
!nvidia-smi

Wed Oct 21 16:39:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")
%env OUT_ROOT=/content/drive/MyDrive/hust/study_at_hust/semesters/20261/data/project_3
%env LYCOS_CSV=/content/lycos.csv
%env CACHE_DIR=/content
%env CLOSR_DIR=/content/project_3/third_party/CLOSR
import os
OUT = os.environ["OUT_ROOT"]
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
env: OUT_ROOT=/content/drive/MyDrive/project_3_runs
env: LYCOS_CSV=/content/lycos.csv
env: CACHE_DIR=/content
env: CLOSR_DIR=/content/project_3/third_party/CLOSR


In [7]:
# Chạy lại ô này sau mỗi lần `git push` từ laptop
%cd /content
!test -d project_3 && git -C project_3 pull || git clone https://github.com/hoangducanh1865/project_3
%cd /content/project_3
!pip install -q -r requirements.txt
!test -d third_party/CLOSR || git clone -q https://github.com/jackwilkie/CLOSR third_party/CLOSR

/content
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.91 KiB | 4.91 MiB/s, done.
From https://github.com/hoangducanh1865/project_3
   43cc277..9a383bf  main       -> origin/main
Updating 43cc277..9a383bf
Fast-forward
 notebooks/week_4_baseline.ipynb | 154 ++++++++++++++++++++++++++++++++++------
 1 file changed, 134 insertions(+), 20 deletions(-)
/content/project_3


In [8]:
# lycos.csv khoảng 860 MB (liên kết từ README của [16])
!test -f /content/lycos.csv || gdown 1PUMAbjz5L0MKiL3P-bEFTYYOcB6zPYB2 -O /content/lycos.csv
!ls -lh /content/lycos.csv

-rw-r--r-- 1 root root 823M May 29  2025 /content/lycos.csv


## 2. Kiểm tra dữ liệu và chạy thử

In [9]:
import os, sys
os.chdir("/content/project_3"); sys.path.insert(0, "/content/project_3")
from hydra import initialize_config_dir, compose
from src.data import load_csv, class_table

with initialize_config_dir(version_base="1.3", config_dir="/content/project_3/conf"):
    cfg = compose(config_name="config", overrides=["protocol=p0"])
X, y = load_csv(cfg.dataset)
print(X.shape)
class_table(y)

bỏ 0 dòng có NaN/Inf trên tổng 1789954
(1789954, 72)


{'benign': 1341169,
 'bot': 735,
 'ddos': 95683,
 'dos_goldeneye': 6765,
 'dos_hulk': 158988,
 'dos_slowhttptest': 4866,
 'dos_slowloris': 5674,
 'ftp_patator': 8006,
 'heartbleed': 11,
 'portscan': 160106,
 'ssh_patator': 5918,
 'webattack_bruteforce': 1360,
 'webattack_sql_injection': 12,
 'webattack_xss': 661}

Xem tên các lớp ở ô trên rồi chỉnh `HELDOUT` ở bước 5 cho khớp. Mỗi mục khớp không phân biệt hoa thường với một phần tên lớp, và phải khớp đúng một lớp.

In [10]:
# Chạy thử vài giây: nếu ô này lỗi thì các bước sau cũng lỗi
!OUT_ROOT=/content/smoke python -m src.run protocol=p0 model=mlp_msp eval_cap=5000 ++model.epochs=1   # không ghi vào OUT_ROOT

[2026-10-21 16:42:14,936][__main__][INFO] - lớp trong dữ liệu: {'benign': 1341169, 'bot': 735, 'ddos': 95683, 'dos_goldeneye': 6765, 'dos_hulk': 158988, 'dos_slowhttptest': 4866, 'dos_slowloris': 5674, 'ftp_patator': 8006, 'heartbleed': 11, 'portscan': 160106, 'ssh_patator': 5918, 'webattack_bruteforce': 1360, 'webattack_sql_injection': 12, 'webattack_xss': 661}
[2026-10-21 16:42:16,792][__main__][INFO] - known=[np.str_('benign'), np.str_('bot'), np.str_('ddos'), np.str_('dos_goldeneye'), np.str_('dos_hulk'), np.str_('dos_slowhttptest'), np.str_('dos_slowloris'), np.str_('ftp_patator'), np.str_('portscan'), np.str_('ssh_patator'), np.str_('webattack_bruteforce'), np.str_('webattack_xss')] | OOD=[np.str_('heartbleed'), np.str_('webattack_sql_injection')] | train=301949 test=5000 ood=23
[2026-10-21 16:42:22,811][__main__][INFO] - {'protocol': 'p0', 'heldout': None, 'model': 'mlp_msp', 'seed': 0, 'device': 'cuda', 'n_train': 301949, 'n_known_test': 5000, 'n_ood': 23, 'ood_classes': [np.st

## 3. Tái hiện số P0 của [16]
Dùng trọng số có sẵn trong repo của [16], chỉ suy luận. Đối chiếu với README của repo:
- CLOSR: closed-set acc 0,995276; open-set AUC 0,974022; OpenAUC 0,969420
- CLAD: closed-set mean AUROC 0,999855; open-set mean AUROC 0,996627

In [11]:
%cd /content/project_3/third_party/CLOSR
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
!mkdir -p data && cp /content/lycos.csv data/lycos.csv
!python eval_closr.py --device cuda 2>&1 | tee {OUT}/tai_hien_closr.txt
!python eval_clad.py --device cuda 2>&1 | tee {OUT}/tai_hien_clad.txt
%cd /content/project_3

/content/project_3/third_party/CLOSR
Traceback (most recent call last):
  File "/content/project_3/third_party/CLOSR/eval_closr.py", line 272, in <module>
    main()
    ~~~~^^
  File "/content/project_3/third_party/CLOSR/eval_closr.py", line 254, in main
    model = load_model(opt)
  File "/content/project_3/third_party/CLOSR/eval_closr.py", line 132, in load_model
    model, _, _, _, _ = load_checkpoint(
                        ~~~~~~~~~~~~~~~^
        opt.checkpoint_path,
        ^^^^^^^^^^^^^^^^^^^^
        model,
        ^^^^^^
    )
    ^
  File "/content/project_3/third_party/CLOSR/util/checkpoint.py", line 61, in load_checkpoint
    checkpoint = T.load(path, map_location="cpu")
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1578, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust

## 4. P0: chia ngẫu nhiên
OOD là các lớp dưới 100 mẫu, giống [16]. CLAD/CLOSR/HC-CLOSR ở đây huấn luyện ít vòng hơn bản gốc (xem `conf/model/*.yaml`), nên không phải số của bài.

In [12]:
MODELS = "mlp_msp,mlp_knn,xgboost,iforest,autoencoder,clad,closr,hc_closr"   # hc_closr: phương pháp đề xuất
!python -m src.run -m protocol=p0 model={MODELS}

[2026-10-21 16:43:45,299][HYDRA] Launching 7 jobs locally
[2026-10-21 16:43:45,299][HYDRA] 	#0 : protocol=p0 model=mlp_msp
[2026-10-21 16:43:48,908][__main__][INFO] - lớp trong dữ liệu: {'benign': 1341169, 'bot': 735, 'ddos': 95683, 'dos_goldeneye': 6765, 'dos_hulk': 158988, 'dos_slowhttptest': 4866, 'dos_slowloris': 5674, 'ftp_patator': 8006, 'heartbleed': 11, 'portscan': 160106, 'ssh_patator': 5918, 'webattack_bruteforce': 1360, 'webattack_sql_injection': 12, 'webattack_xss': 661}
[2026-10-21 16:43:51,102][__main__][INFO] - known=[np.str_('benign'), np.str_('bot'), np.str_('ddos'), np.str_('dos_goldeneye'), np.str_('dos_hulk'), np.str_('dos_slowhttptest'), np.str_('dos_slowloris'), np.str_('ftp_patator'), np.str_('portscan'), np.str_('ssh_patator'), np.str_('webattack_bruteforce'), np.str_('webattack_xss')] | OOD=[np.str_('heartbleed'), np.str_('webattack_sql_injection')] | train=301949 test=50000 ood=23
[2026-10-21 16:43:54,896][__main__][INFO] - {'protocol': 'p0', 'heldout': None, 

## 5. P1: loại từng họ tấn công khỏi tập huấn luyện
- Biến thể của chiến thuật đã biết: `slowloris` (còn các DoS khác), `ssh` (còn FTP-Patator).
- Chiến thuật mới: `bot` (Botnet), `portscan`.

In [13]:
HELDOUT = "slowloris,ssh,bot,portscan"
!python -m src.run -m protocol=p1 protocol.heldout={HELDOUT} model={MODELS}

[2026-10-21 16:52:17,418][HYDRA] Launching 28 jobs locally
[2026-10-21 16:52:17,418][HYDRA] 	#0 : protocol=p1 protocol.heldout=slowloris model=mlp_msp
[2026-10-21 16:52:18,974][__main__][INFO] - lớp trong dữ liệu: {'benign': 1341169, 'bot': 735, 'ddos': 95683, 'dos_goldeneye': 6765, 'dos_hulk': 158988, 'dos_slowhttptest': 4866, 'dos_slowloris': 5674, 'ftp_patator': 8006, 'heartbleed': 11, 'portscan': 160106, 'ssh_patator': 5918, 'webattack_bruteforce': 1360, 'webattack_sql_injection': 12, 'webattack_xss': 661}
[2026-10-21 16:52:20,939][__main__][INFO] - known=[np.str_('benign'), np.str_('bot'), np.str_('ddos'), np.str_('dos_goldeneye'), np.str_('dos_hulk'), np.str_('dos_slowhttptest'), np.str_('ftp_patator'), np.str_('portscan'), np.str_('ssh_patator'), np.str_('webattack_bruteforce'), np.str_('webattack_xss')] | OOD=[np.str_('dos_slowloris')] | train=299395 test=50000 ood=5674
[2026-10-21 16:52:23,522][__main__][INFO] - {'protocol': 'p1', 'heldout': 'slowloris', 'model': 'mlp_msp', 's

## 6. Tổng hợp

In [14]:
!python -m src.report

# Kết quả baseline tuần 4

Số lần chạy: 36. Điểm cao hơn là bất thường hơn; OOD là lớp dương.
FPR95: tỉ lệ known-test báo nhầm khi bắt được 95% OOD. Tỉ lệ phát hiện OOD và FPR tại ngưỡng: ngưỡng chọn từ phân vị 95 của dữ liệu known (val), không dùng dữ liệu OOD.

## P0 – OOD: heartbleed, webattack_sql_injection

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| mlp_msp     |  0.5353 |  1      |                0.5217 |           0.0513 |   0.9946 |     0.8418 |                        3.7 |
| mlp_knn     |  0.9295 |  0.1749 |                0.4783 |           0.0509 |   0.9946 |     0.8418 |                        1.3 |
| xgboost     |  0.99   |  0.0142 |                1      |           0.0506 |   0.9989 |     0.9324 |                       14.4 |
| iforest     |  0.7651 | 

In [15]:
from IPython.display import Markdown
import os
Markdown(open(os.path.join(os.environ["OUT_ROOT"], "bao_cao_tuan4.md")).read())

# Kết quả baseline tuần 4

Số lần chạy: 36. Điểm cao hơn là bất thường hơn; OOD là lớp dương.
FPR95: tỉ lệ known-test báo nhầm khi bắt được 95% OOD. Tỉ lệ phát hiện OOD và FPR tại ngưỡng: ngưỡng chọn từ phân vị 95 của dữ liệu known (val), không dùng dữ liệu OOD.

## P0 – OOD: heartbleed, webattack_sql_injection

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| mlp_msp     |  0.5353 |  1      |                0.5217 |           0.0513 |   0.9946 |     0.8418 |                        3.7 |
| mlp_knn     |  0.9295 |  0.1749 |                0.4783 |           0.0509 |   0.9946 |     0.8418 |                        1.3 |
| xgboost     |  0.99   |  0.0142 |                1      |           0.0506 |   0.9989 |     0.9324 |                       14.4 |
| iforest     |  0.7651 |  0.4988 |                0.4783 |           0.0505 | nan      |   nan      |                        0.9 |
| autoencoder |  0.7171 |  0.5587 |                0.4783 |           0.0511 | nan      |   nan      |                        0.5 |
| clad        |  0.7379 |  0.2697 |                0      |           0.0489 | nan      |   nan      |                       39.3 |
| closr       |  0.9424 |  0.1995 |                0.7826 |           0.05   |   0.9884 |     0.7725 |                      416.8 |
| mlp_msp     |  0.5099 |  1      |                0.5217 |           0.0506 |   0.9906 |     0.7025 |                        5.9 |

## P1 – OOD: bot

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| mlp_msp     |  0.8133 |  1      |                0      |           0.049  |   0.9971 |     0.8086 |                        1.3 |
| mlp_knn     |  0.871  |  0.1425 |                0.0707 |           0.0518 |   0.9971 |     0.8086 |                        1.3 |
| xgboost     |  0.9705 |  0.0536 |                0.932  |           0.0498 |   0.9988 |     0.8896 |                       14.3 |
| iforest     |  0.4854 |  0.5537 |                0.0599 |           0.0507 | nan      |   nan      |                        0.4 |
| autoencoder |  0.4773 |  0.5662 |                0.0612 |           0.0496 | nan      |   nan      |                        0.4 |
| clad        |  0.6719 |  0.3762 |                0      |           0.0497 | nan      |   nan      |                       44.7 |
| closr       |  0.9284 |  0.0741 |                0.019  |           0.0501 |   0.9902 |     0.803  |                      370.5 |

## P1 – OOD: portscan

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| mlp_msp     |  0.963  |  0.0813 |                0.714  |           0.0473 |   0.9947 |     0.8135 |                        0.9 |
| mlp_knn     |  0.8902 |  0.191  |                0.1264 |           0.0522 |   0.9947 |     0.8135 |                        1   |
| xgboost     |  0.9976 |  0.0037 |                1      |           0.0481 |   0.9988 |     0.8984 |                       10.8 |
| iforest     |  0.6383 |  0.3849 |                0      |           0.0519 | nan      |   nan      |                        0.3 |
| autoencoder |  0.8766 |  0.1572 |                0.062  |           0.0504 | nan      |   nan      |                        0.4 |
| clad        |  0.6579 |  0.3741 |                0.0003 |           0.0513 | nan      |   nan      |                       44.6 |
| closr       |  0.9797 |  0.0384 |                0.9967 |           0.0483 |   0.9914 |     0.7624 |                      370.2 |

## P1 – OOD: dos_slowloris

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| mlp_msp     |  0.8817 |  0.5098 |                0.7608 |           0.0492 |   0.9954 |     0.8222 |                        2.5 |
| mlp_knn     |  0.9276 |  0.1356 |                0.5643 |           0.0506 |   0.9954 |     0.8222 |                        1.3 |
| xgboost     |  0.9931 |  0.0233 |                0.9996 |           0.0492 |   0.9989 |     0.9232 |                       14   |
| iforest     |  0.8184 |  0.4826 |                0.5256 |           0.0483 | nan      |   nan      |                        0.4 |
| autoencoder |  0.9072 |  0.4066 |                0.3824 |           0.0482 | nan      |   nan      |                        0.5 |
| clad        |  0.7674 |  0.3718 |                0.0021 |           0.0498 | nan      |   nan      |                       44.8 |
| closr       |  0.9404 |  0.1942 |                0.73   |           0.0505 |   0.9918 |     0.7983 |                      370.5 |

## P1 – OOD: ssh_patator

| model       |   AUROC |   FPR95 |   Tỉ lệ phát hiện OOD |   FPR tại ngưỡng |      Acc |   Macro-F1 |   Thời gian huấn luyện (s) |
|:------------|--------:|--------:|----------------------:|-----------------:|---------:|-----------:|---------------------------:|
| iforest     |  0.7209 |  0.2885 |                0      |           0.0494 | nan      |   nan      |                        0.3 |
| autoencoder |  0.8815 |  0.1207 |                0.0027 |           0.0484 | nan      |   nan      |                        0.5 |
| clad        |  0.7291 |  0.2729 |                0      |           0.0477 | nan      |   nan      |                       44.8 |
| closr       |  0.939  |  0.0688 |                0.0193 |           0.0483 |   0.99   |     0.7519 |                      370.5 |
| mlp_msp     |  0.9625 |  0.0416 |                0.9821 |           0.0481 |   0.9971 |     0.8277 |                        1.2 |
| mlp_knn     |  0.9706 |  0.0299 |                0.9936 |           0.049  |   0.9971 |     0.8277 |                        1.3 |
| xgboost     |  0.9869 |  0.0167 |                1      |           0.0465 |   0.9987 |     0.9155 |                       14.2 |

## P1: AUROC theo họ tấn công bị loại

| model       |    bot |   portscan |   slowloris |    ssh |
|:------------|-------:|-----------:|------------:|-------:|
| autoencoder | 0.4773 |     0.8766 |      0.9072 | 0.8815 |
| clad        | 0.6719 |     0.6579 |      0.7674 | 0.7291 |
| closr       | 0.9284 |     0.9797 |      0.9404 | 0.939  |
| iforest     | 0.4854 |     0.6383 |      0.8184 | 0.7209 |
| mlp_knn     | 0.871  |     0.8902 |      0.9276 | 0.9706 |
| mlp_msp     | 0.8133 |     0.963  |      0.8817 | 0.9625 |
| xgboost     | 0.9705 |     0.9976 |      0.9931 | 0.9869 |


Kết quả nằm ở `OUT_ROOT` trên Drive: `bao_cao_tuan4.md`, `results.csv`, `tai_hien_*.txt` và từng `result.json`. Gửi `bao_cao_tuan4.md` để viết báo cáo.